<a href="https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic3/3.2_database_search_and_vector_stores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# LLM Engineering Essentials от Nebius Academy
Курс на github: [ссылка](https://github.com/Nebius-Academy/LLM-Engineering-Essentials/tree/main)
Курс сейчас находится в разработке, скоро появятся дополнительные материалы.

# Основы LLM Engineering 3.2. Поиск по базе данных и векторным хранилищам
В предыдущем блокноте мы использовали веб-поиск для предоставления дополнительного контекста LLM, но часто соответствующая информация не просто находится в Интернете, а содержится во внутренних базах данных вашей компании.
Из этой тетради вы узнаете, как реализовать RAG с двумя популярными типами баз данных: реляционными базами данных и векторными хранилищами. Другой вариант — графовые базы данных — будет рассмотрен в следующей тетради.
<center>
<img src="https://drive.google.com/uc?export=view&id=1d4fUfWEYn6X1XW-B5VJNy5e5DZxZ30CC" width=600 />
</center>
**Примечание**. Мы по-прежнему будем использовать API-интерфейсы LLM, но во многих реальных ситуациях это не будет хорошим вариантом, поскольку вы не хотите предоставлять свои внутренние данные стороннему API. В таких случаях на помощь приходят самостоятельные LLM; далее в курсе мы научимся их использовать.

## Готовим вещи

In [1]:
!pip install -q openai

In [2]:
# Подставьте свой API-ключ (и при необходимости base_url вашего провайдера)
api_key = "your-api-key-here"
base_url = None  # например: "https://api.openai.com/v1/"

# Опционально: загрузка из файла (если загрузили api_key или openai_api_key в рабочую директорию)
from pathlib import Path
if Path("api_key").exists():
    api_key = Path("api_key").read_text().strip()
elif Path("openai_api_key").exists():
    api_key = Path("openai_api_key").read_text().strip()
if base_url:
    base_url = base_url.rstrip("/") + "/"


# Поиск по базе данных
Во многих приложениях вам необходимо предоставить LLM информацию о различных технических характеристиках продукта, которая хранится в некоторой внутренней базе данных.

# Часть 1: Преобразование текста в SQL
Во многих случаях источником данных будет **реляционная база данных**. Затем **инструмент запросов** должен будет переформулировать первоначальный запрос пользователя в виде SQL-запроса. Примером может быть:
**Запрос пользователя**: `Er, I'd like buy some potions, but I'm low on cash. What are the most affordable options?`
**SQL-запрос**:
```sql
SELECT item_name, price
FROM shop_inventory
WHERE category = 'potion'
ORDER BY price ASC;
```
Задача генерации SQL-запросов на основе запросов пользователей известна как **Text-to-SQL**. Это непросто! Если вы хотите проверить, насколько хороши LLM и системы на его основе, мы рекомендуем проверить таблицы лидеров [BIRD-SQL Big Bench](https://bird-bench.github.io/) и [BIRD-CRITIC](https://bird-critic.github.io/). Второй сложнее и оценивает только чистые LLM, а не системы на базе LLM, и показывает довольно печальные результаты; но даже для систем BIRD-SQL AI точность ниже 80%.
Чтобы компенсировать этот недостаток, системы преобразования текста в SQL часто вводят цепочки проверки и отладки, которые улучшают первоначальный запрос до тех пор, пока он не заработает. Вот, например, схема, описывающая структуру [Open Data QnA]([[PH_00004]]] от Google Cloud Plaftorm.
<center>
<img src="https://raw.githubusercontent.com/GoogleCloudPlatform/Open_Data_QnA/refs/heads/main/utilities/imgs/OpenDataQnA_architecture.png" width=800 />
[Источник](https://github.com/GoogleCloudPlatform/Open_Data_QnA)
</center>
Рекомендуем вам просмотреть несколько лучших решений из таблицы лидеров BIRD и принять к сведению их системные промпты. Здесь мы дадим только общие советы. Типичное приглашение преобразования текста в SQL содержит:
* Набор инструкций, чем подробнее, тем лучше. Некоторые инструкции могут объяснять природу баз данных или некоторые типичные случаи использования; другие устанавливают **ограничения**, обычно запрещая LLM генерировать запросы на изменение таблицы (INSERT, UPDATE, DELETE, DROP и т. д.).
  Обратите внимание, что даже несмотря на эти ограничения, у вас должны быть дополнительные проверки после LLM (на основе регулярных выражений или других LLM), чтобы избежать повреждения ваших таблиц в результате галлюцинации или взлома.
* Схемы базы данных всех необходимых таблиц.
* Если вы можете заранее предсказать некоторые частые случаи использования, отображение их в системном промпте также не повредит.
Давайте проверим пример!

## Пример: бот-помощник в магазине зелий
Мы будем работать с простой базой данных SQL, состоящей из трех таблиц:
```sql
CREATE TABLE shop_inventory (
  potion_id INTEGER PRIMARY KEY,      -- Unique ID of the potion
  stock INTEGER,                      -- How many are in stock
  price INTEGER                       -- Price in gold
)
CREATE TABLE potions (
  potion_id INTEGER PRIMARY KEY,      -- Matches inventory ID
  potion_name TEXT,                   -- Name of the potion
  category TEXT,                      -- Category (healing, mana, etc.)
  effect TEXT,                        -- Specific effect (heals 10 hp, etc.)
  rarity TEXT,                        -- common, uncommon, rare, legendary
  duration TEXT,                      -- How long it lasts (e.g., '1 min', '10 min', 'permanent')
  side_effects TEXT                   -- Possible side effects (nullable)
)
CREATE TABLE purchases (
  purchase_id INTEGER PRIMARY KEY,
  customer_name TEXT,                 -- Name of the customer
  potion_id INTEGER,                  -- Purchased potion
  quantity INTEGER,                   -- Number bought
  date DATE,                          -- Date of purchase
  FOREIGN KEY(potion_id) REFERENCES potions(potion_id)
)
```
Следующий код загрузит эти базы данных вместе с некоторыми полезными сценариями загрузчика и запросов.

In [3]:
!wget https://github.com/Nebius-Academy/LLM-Engineering-Essentials/raw/main/topic3/potion_shop.db -O potion_shop.db
!wget https://github.com/Nebius-Academy/LLM-Engineering-Essentials/raw/main/topic3/potion_shop_utils.py -O potion_shop_utils.py

--2025-04-23 01:56:23--  https://github.com/Nebius-Academy/LLM-Engineering-Essentials/raw/main/topic3/potion_shop.db
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Nebius-Academy/LLM-Engineering-Essentials/main/topic3/potion_shop.db [following]
--2025-04-23 01:56:24--  https://raw.githubusercontent.com/Nebius-Academy/LLM-Engineering-Essentials/main/topic3/potion_shop.db
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16384 (16K) [application/octet-stream]
Saving to: ‘potion_shop.db’

potion_shop.db      100%[===================>]  16.00K  --.-KB/s    in 0s      

2025-04-23 01:56:24 (74.1 MB/s)

Давайте загрузим базу данных и выполним SQL-запрос, чтобы получить три самых дорогих зелья:

In [17]:
from potion_shop_utils import (
    create_potion_shop_database,
    show_schema,
    show_table,
    query_db
)

# Create the database
conn = create_potion_shop_database()

# Display schema
show_schema(conn)

# Display all tables
show_table(conn, "potions")
show_table(conn, "shop_inventory")
show_table(conn, "purchases")

# Example query
print("\nExample query: Most expensive potions")
result = query_db(conn, """
SELECT p.potion_name, s.price
FROM potions p
JOIN shop_inventory s ON p.potion_id = s.potion_id
ORDER BY s.price DESC
LIMIT 3
""")

for name, price in result:
    print(f"{name}: {price} gold")

# Close the connection
conn.close()

Database already exists at potion_shop.db

== Database Schema ==

Table: shop_inventory
  potion_id (INTEGER) [PRIMARY KEY]
  stock (INTEGER)
  price (INTEGER)

Table: potions
  potion_id (INTEGER) [PRIMARY KEY]
  potion_name (TEXT)
  category (TEXT)
  effect (TEXT)
  rarity (TEXT)
  duration (TEXT)
  side_effects (TEXT)

Table: purchases
  purchase_id (INTEGER) [PRIMARY KEY]
  customer_name (TEXT)
  potion_id (INTEGER)
  quantity (INTEGER)
  date (DATE)

Table: potions
    potion_id                potion_name    category  \
0           1       Minor Healing Potion     healing   
1           2             Healing Potion     healing   
2           3       Major Healing Potion     healing   
3           4          Minor Mana Potion        mana   
4           5                Mana Potion        mana   
5           6          Major Mana Potion        mana   
6           7      Crude Barkskin Potion  protection   
7           8    Refined Barkskin Potion  protection   
8           9        

Это кажется правильным.
Теперь мы создадим класс `TextToSQLRAGBot`, реализующий бота на базе RAG, который для каждого запроса пользователя:
* Создает SQL-запрос. Для этого мы по умолчанию используем модель **Phi-4** my Microsoft, которая является одной из лучших моделей с открытым исходным кодом в таблице лидеров [BIRD-CRITIC](https://bird-critic.github.io/).
* Запускает этот SQL-запрос.
* Формулирует удобный для человека ответ на основе запроса пользователя и всего, что было получено из базы данных. Для этого мы используем по умолчанию **Llama-3.1-70B**, но, конечно, подойдет и гораздо меньшая модель.
Мы решили, что простой сценарий не требует реализации многоходовых разговоров, поэтому у бота нет памяти разговоров.
И прежде всего, давайте создадим системный промпт для генерации SQL в соответствии с рекомендациями, описанными выше. Обратите внимание, что это не системный запрос в строгом смысле слова, поскольку вопрос пользователя будет его частью как **{question}**:

In [5]:
text_to_sql_system_prompt = """**Task**

Generate a SQL query to answer the user's question. The data comes from a fantasy role-playing game shop system.

**Instructions**

- Generate a syntactically correct SQL query using only the schema provided.
- Do **not** select all columns with `SELECT *`; only select relevant columns.
- If the user requests sorted results, use `ORDER BY`—otherwise, avoid it.
- Do **not** return columns the user did not explicitly ask for.
- If a question cannot be answered with the available schema, return: `'I do not know'`.
- Do **not** use DML statements (INSERT, UPDATE, DELETE, DROP, etc.).
- Always use meaningful table aliases when joining multiple tables.
- You may assume `gold` is the unit of currency.

**Database Schema**

This query will run on a database with the following schema:

```sql
CREATE TABLE shop_inventory (
  potion_id INTEGER PRIMARY KEY,      -- Unique ID of the potion
  stock INTEGER,                      -- How many are in stock
  price INTEGER                       -- Price in gold
)

CREATE TABLE potions (
  potion_id INTEGER PRIMARY KEY,      -- Matches inventory ID
  potion_name TEXT,                   -- Name of the potion
  category TEXT,                      -- Category (healing, mana, etc.)
  effect TEXT,                        -- Specific effect (heals 10 hp, etc.)
  rarity TEXT,                        -- common, uncommon, rare, legendary
  duration TEXT,                      -- How long it lasts (e.g., '1 min', '10 min', 'permanent')
  side_effects TEXT                   -- Possible side effects (nullable)
)

CREATE TABLE purchases (
  purchase_id INTEGER PRIMARY KEY,
  customer_name TEXT,                 -- Name of the customer
  potion_id INTEGER,                  -- Purchased potion
  quantity INTEGER,                   -- Number bought
  date DATE,                          -- Date of purchase
  FOREIGN KEY(potion_id) REFERENCES potions(potion_id)
)
```

**Your Output**

Given the schema above, return the correct SQL query to answer this question:
**‘{question}’**

```sql
"""

In [6]:
from typing import Dict, Any, Optional, Callable
import sqlite3
import pandas as pd
import re

class TextToSQLRAGBot:
    def __init__(
        self,
        sql_client,  # Client for the SQL generation model
        response_client,  # Client for the response generation model
        database_connection: sqlite3.Connection,
        sql_model: str = "microsoft/phi-4",
        response_model: str = "meta-llama/Meta-Llama-3.1-70B-Instruct",
        get_sql_system_prompt = text_to_sql_system_prompt,
        get_response_system_prompt = None
    ):
        """Initialize the text-to-SQL RAG bot.

        Args:
            sql_client: Client for the SQL generation model (phi-4)
            response_client: Client for the response generation model (Llama-70b)
            database_connection: SQLite database connection
            sql_model: The model to use for SQL generation
            response_model: The model to use for response generation
            get_sql_system_prompt: Function to retrieve the system message for SQL generation
            get_response_system_prompt: Function to retrieve the system message for response generation
        """
        self.sql_client = sql_client
        self.response_client = response_client
        self.sql_model = sql_model
        self.response_model = response_model
        self.db_conn = database_connection

        self.get_sql_system_prompt = get_sql_system_prompt

        # Default system message for response generation if none is provided
        if get_response_system_prompt is None:
            self.get_response_system_prompt = lambda: """You are a helpful potion shop assistant that provides information based on database query results.

You will be given:
1. The original user question
2. The SQL query that was generated to answer the question
3. The results of executing that query on the potion shop database

Formulate a natural, helpful response that answers the user's question based on the query results.
Speak as if you are a knowledgeable potion shop employee helping a customer.
If the query returned no results, explain that to the user in a friendly way.
"""
        else:
            self.get_response_system_prompt = lambda: get_response_system_prompt

    def generate_sql_query(self, user_question: str) -> str:
        """Generate an SQL query from a natural language question.

        Args:
            user_question: The natural language question from the user

        Returns:
            str: The generated SQL query
        """

        try:
            # Get SQL query from the model
            completion = self.sql_client.chat.completions.create(
                model=self.sql_model,
                messages=[{
                    "role": "user",
                    "content": self.get_sql_system_prompt.format(question=user_question)
                }]
            )

            response = completion.choices[0].message.content

            # Look for SQL code blocks
            sql_blocks = re.findall(r"```sql\s*(.*?)\s*```", response, re.DOTALL)
            if sql_blocks:
                sql_query = sql_blocks[-1].strip()
            else:
                # Look for generic code blocks
                code_blocks = re.findall(r"```\s*(.*?)\s*```", response, re.DOTALL)
                if code_blocks:
                    sql_query = code_blocks[-1].strip()
                else:
                    # If no SQL-specific blocks, look for generic code blocks
                    # Which will most likely result in a failure
                    sql_query = response

            return sql_query

        except Exception as e:
            return f"Error generating SQL query: {str(e)}"

    def execute_query(self, sql_query: str) -> (pd.DataFrame, str):
        """Execute the SQL query on the database.

        Args:
            sql_query: The SQL query to execute

        Returns:
            tuple: (DataFrame with results, error message if any)
        """
        try:
            # Execute query and return results as a pandas DataFrame
            results = pd.read_sql_query(sql_query, self.db_conn)
            return results, None
        except Exception as e:
            return None, f"Error executing SQL query: {str(e)}"

    def generate_response(self, user_question: str, sql_query: str, query_results: pd.DataFrame) -> str:
        """Generate a natural language response based on the query results.

        Args:
            user_question: The original user question
            sql_query: The SQL query that was executed
            query_results: The results of the query as a pandas DataFrame

        Returns:
            str: The natural language response
        """
        messages = []
        system_prompt = self.get_response_system_prompt()
        if system_prompt:
            messages.append({
                "role": "system",
                "message": system_prompt
                })

        # Convert DataFrame to string representation
        if query_results is not None:
            results_str = query_results.to_string()
        else:
            results_str = "No results returned (query may have failed)"

        messages.append({
            "role": "user",
            "content": f"""User question: {user_question}

SQL query used:
{sql_query}

Query results:
{results_str}

Please provide a helpful response based on these results."""
        })

        try:
            # Get response from the model
            completion = self.response_client.chat.completions.create(
                model=self.response_model,
                messages=messages
            )

            response = completion.choices[0].message.content
            return response

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def chat(self, user_question: str) -> Dict[str, Any]:
        """Process a user question end-to-end.

        Args:
            user_question: The natural language question from the user

        Returns:
            dict: Dictionary containing the original question, generated SQL, query results, and final response
        """
        # Step 1: Generate SQL query from user question
        sql_query = self.generate_sql_query(user_question)

        # Step 2: Execute the SQL query on the database
        query_results, error = self.execute_query(sql_query)

        # Step 3: Generate a natural language response based on the results
        if error:
            response = f"I'm sorry, but I couldn't execute the query. {error}"
        else:
            response = self.generate_response(user_question, sql_query, query_results)

        # Return a dictionary with all components of the process
        return {
            "user_question": user_question,
            "generated_sql": sql_query,
            "query_results": query_results,
            "response": response
        }

Теперь давайте создадим экземпляр RAG-бота и выполним несколько простых SQL-запросов:

In [25]:
from openai import OpenAI
import sqlite3


client = OpenAI(api_key=api_key, base_url=base_url or None)
)
sql_model = "microsoft/phi-4"
response_model = "meta-llama/Meta-Llama-3.1-70B-Instruct"

sql_client = client
response_client = client

# Create or load the database
db_conn = create_potion_shop_database()

# Initialize the Text-to-SQL RAG bot
text_to_sql_bot = TextToSQLRAGBot(
    sql_client=client,
    response_client=client,
    database_connection=db_conn,
    sql_model=sql_model,
    response_model=response_model
)

# Example questions to test the bot
example_questions = [
    "What are the most expensive potions?",
    "Which customer has spent the most money at the shop?",
    "How many healing potions do we have in stock?",
    "What potions has Alaric purchased?",
    "Which potions cause side effects related to dizziness or headaches?"
]

# Process each question
for i, question in enumerate(example_questions):
    print(f"\n--- Example {i+1}: {question} ---")

    result = text_to_sql_bot.chat(question)

    print("\nGenerated SQL:")
    print(result["generated_sql"])

    print("\nQuery Results:")
    if result["query_results"] is not None:
        print(result["query_results"])
    else:
        print("No results (query may have failed)")

    print("\nResponse:")
    print(result["response"])

    print("\n" + "="*60)



Database already exists at potion_shop.db

--- Example 1: What are the most expensive potions? ---

Generated SQL:
SELECT p.potion_name, s.price
FROM shop_inventory s
JOIN potions p ON s.potion_id = p.potion_id
ORDER BY s.price DESC;

Query Results:
                  potion_name  price
0     Mighty Berserker Elixir     70
1       Superior Speed Potion     65
2        Major Healing Potion     60
3     Dreamless Sleep Draught     50
4           Major Mana Potion     45
5     Refined Barkskin Potion     40
6   Frenzied Berserker Elixir     35
7          Swift Speed Potion     30
8              Healing Potion     25
9               Clarity Tonic     25
10                Mana Potion     20
11                   Antidote     20
12      Crude Barkskin Potion     15
13       Minor Healing Potion     10
14          Minor Mana Potion      8

Response:
The top 5 most expensive potions are:

1. Mighty Berserker Elixir ($70)
2. Superior Speed Potion ($65)
3. Major Healing Potion ($60)
4. Dreamless S

Не стесняйтесь проверить, правильно ли ответил наш бот!

Теперь менее удачный пример:

In [29]:
result = text_to_sql_bot.chat("""
Today is 2024-05-01 (May 1st, 2024)
How many of each of the following potions we had on 2024-02-01
if we didn't have resupply since January:
- Minor Healing Potion
- Major Mana Potion
- Swift Speed Potion?
""")

In [31]:
print(result["generated_sql"])

SELECT 
    pot.potion_name,
    (inv.stock - COALESCE(SUM(pur.quantity), 0)) AS remaining_stock
FROM 
    potions pot
JOIN 
    shop_inventory inv ON pot.potion_id = inv.potion_id
LEFT JOIN 
    purchases pur ON pot.potion_id = pur.potion_id
    AND pur.date < '2024-02-01'
WHERE 
    pot.potion_name IN ('Minor Healing Potion', 'Major Mana Potion', 'Swift Speed Potion')
GROUP BY 
    pot.potion_name, inv.stock;


Здесь проблема в том, что вместо добавления всех покупок, совершенных с 1 февраля 2024 года, к текущим запасам, LLM предлагает вычесть из сегодняшних запасов покупки, которые произошли *до* 01 февраля 2024 года. Эту проблему можно было бы решить с помощью более мощной LLM: Клод хорошо справляется с этим запросом.

In [39]:
# Uncomment this to close the database connection when done
# db_conn.close()

## За пределами SQL
Реляционные базы данных отлично подходят для работы со структурированными данными. Однако иногда вам просто нужно хранить кучу текста, или кода, или JSON, или, что еще хуже, изображений или видео. Давайте посмотрим, как с ними бороться!

# Часть 2: Векторные базы данных
Мы можем хранить все, что захотим, в формате `(key, value)` при условии, что ключи удобны для извлечения. И оказывается, что векторы действительно хороши в этой роли!
Итак, **база данных векторов** (или **хранилище векторов**) хранит данные в формате «ключ-значение» с **векторами для ключей**. Давайте кратко обсудим общий процесс использования векторной базы данных.

## Кодеры и встраивания
Чтобы поместить объект в векторное хранилище, вы должны **закодировать** его как вектор, или, можно также сказать, вы создаете его **(векторное) встраивание**. Это делается с помощью специального **кодировщика**, который обычно представляет собой своего рода нейронную сеть. Вы можете узнать больше о **текстовых кодировщиках** в [специальном длинном чтении](https://nebius-academy.github.io/knowledge-base/text-encoders/). Также существует ряд популярных моделей встраивания изображений и видео (скоро мы также напишем о них длинное чтение).
Очень важно отметить, что хороший кодер не просто преобразует объекты в векторы каким-то случайным образом. Скорее, важнейшей особенностью хорошего кодировщика является то, что **он отображает семантически близкие объекты в геометрически близкие векторы**. Так, на рисунке ниже векторы, изображающие орков, группируются вместе, а векторы, изображающие кошек, находятся на расстоянии от них.
<center>
<img src="https://drive.google.com/uc?export=view&id=14m2FfYjE_XUXGJkc4D8w4zzAlxVNVBrb" width=600 />
</center>
В качестве меры геометрической близости используется либо **Евклидово расстояние**, либо **косинусное подобие**, хотя иногда из-за его вычислительной простоты используется просто скалярное произведение $(x, y) = x^Ty = x_1y_1+\ldots+x_dy_d$.
<center>
<img src="https://drive.google.com/uc?export=view&id=1srHjEQuiGPoLaz9KEt8IXRxZol2mJjqG" width=600 />
</center>

## Поиск в векторных магазинах
Поиск в векторном хранилище основан на механизме **Поиска ближайшего соседа**. Запрос к векторной базе данных — это объект, подобный тем, которые хранятся в этой базе данных. Что происходит с запросом:
1. Сначала запрос векторизуется кодировщиком для его внедрения.
2. Затем база данных находит заданное количество векторов ближайших соседей.
3. Наконец, база данных возвращает объекты, соответствующие этим векторам.
<center>
<img src="https://drive.google.com/uc?export=view&id=1ZeUSt-_VsBRZtHCfFNzde6iwiM9ltBgx" width=600 />
</center>
**Примечание**. Такие модели, как **CLIP**, способны последовательно кодировать как тексты, так и изображения, сопоставляя семантически связанные тексты и изображения с геометрически близкими векторами. С помощью такой модели вы можете, например, иметь базу данных изображений и запрашивать ее с помощью текста. Вы попробуете это в практической части.

Конечно, эффективный поиск требует быстрого поиска ближайшего соседа, а для больших баз данных и/или сценариев с низкой задержкой не рекомендуется просто циклически перебирать все элементы в поисках ближайшего. Итак, существует ряд оптимизированных стратегий, которые предварительно вычисляют некоторые дополнительные структуры данных, которые позволяют ускорить (пусть даже приблизительный) поиск соседей. Один из самых популярных — **HNSW** (**Иерархический навигационный маленький мир**). Если вам интересно, в этом блокноте есть краткое объяснение и небольшая демонстрация.

##Векторный магазин зоопарк
Сейчас доступно огромное разнообразие векторных баз данных, включая [Chroma](https://docs.trychroma.com/docs/overview/getting-started), [LanceDB](https://lancedb.github.io/lancedb/), [Weaviate](https://weaviate.io/developers/weaviate), [Qdrant](https://qdrant.tech/documentation/) и многие, многие другие — и выбор одной — непростая задача. Когда у вас меньше нескольких миллионов векторов для хранения, на самом деле это скорее вопрос удобства разработчика. Хотя в более крупных масштабах скорость предварительных вычислений и поиска ближайшего соседа может начать существенно влиять на общую эффективность.
Если вам нужно выбрать векторное хранилище для вашего проекта, вы можете начать с проверки [этой сравнительной таблицы](https://superlinked.com/vector-db-comparison), или аналогичной.

## RAG с демо-версией LanceDB

В этой демонстрации мы применим RAG для ответа на вопросы о библиотеке `transformers` с использованием документации этой библиотеки.

In [2]:
!pip install -q openai

In [7]:
# Подставьте свой API-ключ (и при необходимости base_url вашего провайдера)
api_key = "your-api-key-here"
base_url = None  # например: "https://api.openai.com/v1/"

# Опционально: загрузка из файла (если загрузили api_key или openai_api_key в рабочую директорию)
from pathlib import Path
if Path("api_key").exists():
    api_key = Path("api_key").read_text().strip()
elif Path("openai_api_key").exists():
    api_key = Path("openai_api_key").read_text().strip()
if base_url:
    base_url = base_url.rstrip("/") + "/"


Давайте документы.

### Предварительная обработка текста
Встраивание магии не будет работать должным образом, если текст загромождён артефактами форматирования. Если форматирование не несет смысла (как в коде), обычно лучше удалить его перед кодированием.
Итак, мы собираемся убрать уценку из большей части его структуры, оставив только простой текст.

In [8]:
import os
import re

from tqdm import tqdm
from bs4 import BeautifulSoup
from markdown import markdown
from pathlib import Path


def markdown_to_text(markdown_string):
    """ Converts a markdown string to plaintext """

    # md -> html -> text since BeautifulSoup can extract text cleanly
    html = markdown(markdown_string)

    html = re.sub(r'<!--((.|\n)*)-->', '', html)
    html = re.sub('<code>bash', '<code>', html)

    # extract text
    soup = BeautifulSoup(html, "html.parser")
    text = ''.join(soup.findAll(text=True))

    text = re.sub('```(py|diff|python)', '', text)
    text = re.sub('```\n', '\n', text)
    text = re.sub('-         .*', '', text)
    text = text.replace('...', '')
    text = re.sub('\n(\n)+', '\n\n', text)

    return text


def prepare_files(input_dir="transformers/docs/source/en/", output_dir="docs"):
    # Convert string paths to Path objects
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    # Check if input directory exists
    assert input_dir.is_dir(), "Input directory doesn't exist"
    output_dir.mkdir(parents=True, exist_ok=True)

    for root, subdirs, files in tqdm(os.walk(input_dir)):
        root_path = Path(root)
        for file_name in files:
            file_path = root_path / file_name
            parent = root_path.stem if root_path.stem != input_dir.stem else ""

            if file_path.is_file():
                with open(file_path, encoding="utf-8") as f:
                    md = f.read()
                text = markdown_to_text(md)

                output_file = output_dir / f"{parent}_{Path(file_name).stem}.txt"
                with open(output_file, "w", encoding="utf-8") as f:
                    f.write(text)


In [4]:
!git clone https://github.com/huggingface/transformers

Cloning into 'transformers'...
remote: Enumerating objects: 278088, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 278088 (delta 80), reused 40 (delta 36), pack-reused 277955 (from 3)
Receiving objects: 100% (278088/278088), 289.12 MiB | 25.67 MiB/s, done.
Resolving deltas: 100% (206671/206671), done.


In [9]:
prepare_files()

0it [00:00, ?it/s]<ipython-input-8-d285d05c636f>:21: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  text = ''.join(soup.findAll(text=True))
6it [00:07,  1.18s/it]


### Разбивка на части
Кодировщики текста могут столкнуться с трудностями при обработке длинных, насыщенных информацией текстовых фрагментов. Чтобы облегчить жизнь кодировщику (и улучшить результаты RAG), вам необходимо разделить контекст на относительно небольшие значимые фрагменты.
Тем не менее, не существует «золотой» стратегии разбиения на блоки. (В идеале, каждый фрагмент должен представлять собой полноценный контент, посвященный одной теме, но это редко работает таким образом.) В общем, используется одна из следующих двух (простых) стратегий:
1. **Фиксированный размер токена с перекрытием**. Это очень простой подход, поскольку он может сбить, обрезать и разделить полезную информацию. Перекрывая фрагменты, мы гарантируем, что важная информация на границах каждого фрагмента будет собрана в нескольких фрагментах, помогая минимизировать потери и поддерживать непрерывность контекста.
Обратите внимание, что размер чанка обычно составляет несколько сотен токенов. Вы можете начать с 512, а затем при необходимости настроить его под свою задачу.
1. **Рекурсивное разбиение**. Прежде всего, для этого необходимо установить максимальную длину фрагмента в токенах. Теперь, в качестве примера, вы можете разделить простой текстовый документ следующим образом:
    - Сначала разбейте фрагмент на '\n\n' (это конец «раздела»).
    - Затем, если «раздел» слишком длинный, на «\n» (по абзацам).
    - Тогда через '.' (по предложению).
    - На этом этапе, если ваша длина все еще превышает максимальную длину, просто разделите ее на фиксированную длину токена.
Обратите внимание, что последовательность разделителей ['\n\n', '\n', '.'] может быть заменена любой другой, подходящей для ваших данных. (Рекурсивный разделитель текста символов Langchain реализует именно эту стратегию.)
Конечно, вы можете использовать другие особенности ваших данных, чтобы разбить их на более содержательные. Например, если вы имеете дело с уценкой, разумно использовать специальные разделители для рекурсивного разделения. Для иллюстрации в Markdown Text Splitter от LangChain используется следующий список:
```
[
    "\\n#{1,6} ",
    "```\\n",
    "\\n\\*\\*\\*+\\n",
    "\\n---+\\n",
    "\\n___+\\n",
    "\\n\\n",
    "\\n",
    " ",
    "",
]
```
Последнее «» указывает, что мы только что разделили токен на фиксированную длину, указанную пользователем при настройке инструмента «Разделение текста», если все разделители исчерпаны.
Хорошей практикой является оценка ваших фрагментов вручную, чтобы проверить, в порядке ли их размер и остаются ли они значимыми при выбранной вами стратегии разделения.

Разбиение на фрагменты мы будем выполнять с помощью рекурсивного разделителя текста из библиотеки `Langchain`.

In [ ]:
!pip install lancedb pyarrow tiktoken -q
!pip install -qU langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.6 MB/s eta 0:00:00


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
        "\n",
        ".",
        " "
    ],
    chunk_size=1024,
    chunk_overlap=128,
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
import os
from typing import List
from functools import partial

import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

### Создание базы данных
Давайте создадим базу данных LanceDB. Для этого нам нужно определить несколько вещей:
1. **Модель встраивания**. Это настроено с
  `embed_func = get_registry().get(<provider name>).create(name=<embedding name>)`
  Например,
  `embed_func = get_registry().get("openai").create(name="text-embedding-ada-002")`
  Для демонстрации мы будем использовать бесплатную модель от Hugging Face.
2. **Схема записи базы данных**. Как минимум он должен содержать:
  * Поле **вектор**
  * **поле векторизованное**, в нашем случае `text`. Он отмечен `embed_func.SourceField()`

In [ ]:
# This line is needed in case you've ran this cell before to clear the db dir
!rm -rf /tmp/lancedb

db = lancedb.connect("/tmp/lancedb")

In [ ]:
import openai
import pyarrow as pa

# We use this model as the encoder: https://huggingface.co/BAAI/bge-small-en-v1.5
embed_func = get_registry().get("huggingface").create(name="BAAI/bge-small-en-v1.5")


class BasicSchema(LanceModel):
    '''
    This is how we store data in the database.
    We need to have a vector here, but apart from this, we may have many other fields
    '''
    text: str = embed_func.SourceField()
    vector: Vector(embed_func.ndims()) = embed_func.VectorField(default=None)

lance_table = db.create_table(
    "transformer_docs",
    mode='overwrite',
    schema=BasicSchema
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Давайте наполним базу данных:

In [ ]:
from tqdm import tqdm
splitted_docs = []

for file in tqdm(os.listdir("docs")):
    with open("docs/"+file, "r") as f:
        text = f.read()
        docs = text_splitter.create_documents([text])
        splitted_docs.extend([{"text": doc.page_content} for doc in docs])

100%|██████████| 514/514 [00:00<00:00, 5246.10it/s]


In [ ]:
print("Total splits:", len(splitted_docs))
print("==First split:==\n", splitted_docs[0])
print("==Second split:==\n", splitted_docs[1])

Total splits: 4423
==First split:==
 {'text': 'Processors\nMultimodal models require a preprocessor capable of handling inputs that combine more than one modality. Depending on the input modality, a processor needs to convert text into an array of tensors, images into pixel values, and audio into an array with tensors with the correct sampling rate.\nFor example, PaliGemma is a vision-language model that uses the SigLIP image processor and the Llama tokenizer. A [ProcessorMixin] class wraps both of these preprocessor types, providing a single and unified processor class for a multimodal model.\nCall [~ProcessorMixin.from_pretrained] to load a processor. Pass the input type to the processor to generate the expected model inputs, input ids and pixel values.'}
==Second split:==
 {'text': 'from transformers import AutoProcessor, PaliGemmaForConditionalGeneration\nfrom PIL import Image\nimport requests\nprocessor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224")\nprompt = "answe

Заполнение таблицы может занять некоторое время:

In [ ]:
lance_table.add(
    splitted_docs,
    on_bad_vectors='drop'  # or 'fill' with fill_value=0.0
)

Мы создадим функцию `search_table`, которая позволит нам получать до `limits` документов из базы данных для нашего `query`:

In [ ]:
def search_table(table, query, max_results=5):
    return table.search(query).limit(max_results).to_pydantic(BasicSchema)

In [ ]:
result = search_table(lance_table, "How to load an LLM in 4 bit quantization?", max_results=5)
result

[BasicSchema(text='Set up a [BitsAndBytesConfig] and set load_in_4bit=True to load a model in 4-bit precision. The [BitsAndBytesConfig] is passed to the quantization_config parameter in [~PreTrainedModel.from_pretrained].\nAllow Accelerate to automatically distribute the model across your available hardware by setting device_map=“auto”.\nPlace all inputs on the same device as the model.\n\nfrom transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM\nquantization_config = BitsAndBytesConfig(load_in_4bit=True)\ntokenizer = AutoTokenizer("meta-llama/Llama-3.1-8B")\nmodel = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B", device_map="auto", quantization_config=quantization_config)\nprompt = "Hello, my llama is cute"\ninputs = tokenizer(prompt, return_tensors="pt").to(model_8bit.device)\ngenerated_ids = model_8bit.generate(**inputs)\noutputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)', vector=FixedSizeList(dim=384)),
 BasicSchema(te

In [ ]:
result = search_table(lance_table, "LLM in 4 bit quantization", max_results=5)
result

[BasicSchema(text='bitsandbytes\nbitsandbytes features the LLM.int8 and QLoRA quantization to enable accessible large language model inference and training.\nLLM.int8() is a quantization method that aims to make large language model inference more accessible without significant degradation. Unlike naive 8-bit quantization, which can result in loss of critical information and accuracy, LLM.int8() dynamically adapts to ensure sensitive components of the computation retain higher precision when needed.\nQLoRA, or 4-bit quantization, compresses a model even further to 4-bits and inserts a small set of trainable low-rank adaptation (LoRA) weights to allowing training. \n\nNote: For a user-friendly quantization experience, you can use the bitsandbytes community space.\n\nRun the command below to install bitsandbytes.', vector=FixedSizeList(dim=384)),
 BasicSchema(text='Quantization Fundamentals with Hugging Face\nQuantization in Depth\nIntroduction to Quantization cooked in 🤗 with 💗🧑\u200d🍳\

Как видите, результаты поиска могут меняться по мере переформулирования запроса.

### Отвечаем на вопросы с помощью RAG
Наконец-то пришло время что-то сгенерировать!
В предыдущем блокноте функция `answer_with_db` использовала веб-поиск; мы обновим его, чтобы вместо этого использовать поиск по базе данных.

In [ ]:
from openai import OpenAI
import os

llm_client = OpenAI(api_key=api_key, base_url=base_url or None)
)
llama_8b_model = "meta-llama/Meta-Llama-3.1-8B-Instruct"

def prettify_string(text, max_line_length=80):
    """Prints a string with line breaks at spaces to prevent horizontal scrolling.
    Args:
        text: The string to print.
        max_line_length: The maximum length of each line.
    """
    output_lines = []
    lines = text.split("\n")
    for line in lines:
        current_line = ""
        words = line.split()
        for word in words:
            if len(current_line) + len(word) + 1 <= max_line_length:
                current_line += word + " "
            else:
                output_lines.append(current_line.strip())
                current_line = word + " "
        output_lines.append(current_line.strip())  # Append the last line
    return "\n".join(output_lines)

def search_result_to_context(search_result):
    return "\n\n".join(
        [record.text for record in search_result]
    )

def answer_with_rag(
    prompt: str,
    system_prompt=None,
    max_tokens=512,
    client=llm_client,
    model=llama_8b_model,
    table=None,
    prettify=True,
    temperature=0.6,
    max_results=5,
    verbose=False
) -> str:
    """
    Generate an answer using RAG (Retrieval-Augmented Generation) with database search.

    Args:
        prompt: User's question or prompt
        system_prompt: Instructions for the LLM
        max_tokens: Maximum number of tokens in the response
        client: OpenAI client instance
        model: Model identifier
        table: LanceDB table
        prettify: Whether to format the output text
        temperature: Temperature for response generation
        max_results: Maximal number of documents to fetch from the table
        verbose: whether to return the search results as well

    Returns:
        Generated response incorporating search results
    """
    # Perform database search
    if table:
        try:
            search_results = search_table(table, prompt, max_results=max_results)
        except:
            search_results = []
    else:
        search_results = []

    # Construct messages with search results
    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    # Add user prompt
    messages.append({
        "role": "user",
        "content":
            f"""Answer the following query using the context provided.

            <context>\n{search_results}\n</context>

            <query>{prompt}</query>
            """
    })

    # Generate completion
    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )

    if prettify:
        answer = prettify_string(completion.choices[0].message.content)
    else:
        answer = completion.choices[0].message.content

    if verbose:
        return {
            "answer": answer,
            "search_results": search_results
        }
    else:
        return answer


Теперь давайте сначала попробуем спросить Ламу, как загрузить LLM в 4-битном квантовании без предоставления контекста (`table=None`).

In [ ]:
client = OpenAI(api_key=api_key, base_url=base_url or None)
)
model = "meta-llama/Meta-Llama-3.1-8B-Instruct"

results = answer_with_rag("""How to load an LLM in 4 bit quantization?""",
               client=client, model=model, table=None, verbose=True)
print(results["answer"])

Unfortunately, there is no context provided to give a specific answer to the
query. However, I can provide a general outline on how to load a Large Language
Model (LLM) in 4-bit quantization.

**4-bit Quantization**

4-bit quantization is a technique used to reduce the precision of neural
network weights and activations from 32-bit floating-point numbers to 4-bit
integers. This reduces the memory footprint and computational requirements of
the model.

**Loading an LLM in 4-bit Quantization**

To load an LLM in 4-bit quantization, you'll need to follow these general
steps:

1. **Quantize the LLM weights**: Use a quantization library or framework (e.g.,
TensorFlow, PyTorch, or ONNX) to convert the LLM weights from 32-bit
floating-point numbers to 4-bit integers. This involves mapping the weight
values to a smaller range of values that can be represented by 4-bit integers.
2. **Choose a quantization strategy**: Select a quantization strategy, such as:
* **Weight-wise quantization**: Quant

Как видите, ответ расплывчатый и не квалифицированный. Теперь давайте добавим правильный контекст!

In [ ]:
client = OpenAI(api_key=api_key, base_url=base_url or None)
)
model = "meta-llama/Meta-Llama-3.1-8B-Instruct"

results = answer_with_rag("""How to load an LLM in 4 bit quantization?""",
               client=client, model=model, table=lance_table, verbose=True)
print(results["answer"])

To load an LLM in 4-bit quantization, you can use the `BitsAndBytesConfig`
class from the `transformers` library. Here is an example of how you can do it:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer,
BitsAndBytesConfig

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained(
"meta-llama/Llama-3.1-8B",
quantization_config=BitsAndBytesConfig(load_in_4bit=True),
device_map="auto"
)
```

In this example, the `BitsAndBytesConfig` class is used to specify the
quantization configuration. The `load_in_4bit=True` parameter tells the model
to load in 4-bit precision. The `device_map="auto"` parameter allows the model
to automatically distribute itself across the available hardware.

Note that you need to make sure that the model you are loading is compatible
with 4-bit quantization. Also, the performance benefits of 4-bit quantization
may vary depending on the specific hardware a

Это намного лучше!

In [ ]:
results["search_results"]

[BasicSchema(text='Set up a [BitsAndBytesConfig] and set load_in_4bit=True to load a model in 4-bit precision. The [BitsAndBytesConfig] is passed to the quantization_config parameter in [~PreTrainedModel.from_pretrained].\nAllow Accelerate to automatically distribute the model across your available hardware by setting device_map=“auto”.\nPlace all inputs on the same device as the model.\n\nfrom transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM\nquantization_config = BitsAndBytesConfig(load_in_4bit=True)\ntokenizer = AutoTokenizer("meta-llama/Llama-3.1-8B")\nmodel = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B", device_map="auto", quantization_config=quantization_config)\nprompt = "Hello, my llama is cute"\ninputs = tokenizer(prompt, return_tensors="pt").to(model_8bit.device)\ngenerated_ids = model_8bit.generate(**inputs)\noutputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)', vector=FixedSizeList(dim=384)),
 BasicSchema(te

# Практическая часть
Если вы столкнулись с какими-либо трудностями или просто хотите увидеть наши решения, загляните в [Блокнот решений](https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic3/3.2_database_search_and_vector_stores_solutuons.ipynb).

## Задача 1. Простая рекомендательная система
Векторный поиск является разумной основой для поиска или рекомендаций. В этой задаче мы протестируем его на простой базе данных, содержащей информацию о товарах, которые может продавать фэнтезийный торговец. (Конечно, это могут быть любые товары, которые продает интернет-магазин.)
Давайте загрузим данные.

In [1]:
!wget https://github.com/Nebius-Academy/LLM-Engineering-Essentials/raw/main/topic3/fantasy_items_descriptions.json -O fantasy_items_descriptions.json

--2025-04-19 01:35:38--  https://github.com/Nebius-Academy/LLM-Engineering-Essentials/raw/main/topic3/fantasy_items_descriptions.json
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Nebius-Academy/LLM-Engineering-Essentials/main/topic3/fantasy_items_descriptions.json [following]
--2025-04-19 01:35:38--  https://raw.githubusercontent.com/Nebius-Academy/LLM-Engineering-Essentials/main/topic3/fantasy_items_descriptions.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 361418 (353K) [text/plain]
Saving to: ‘fantasy_items_descriptions.json’

fantasy_items_descr 100%[===================>] 352.95K  

In [7]:
import json

with open('fantasy_items_descriptions.json', 'r') as f:
    data = json.load(f)

data[0]

{'name': 'Enhanced Minor Healing Potion',
 'category': 'Potion',
 'rarity': 'Uncommon',
 'structured': {'name': 'Enhanced Minor Healing Potion',
  'category': 'Potion',
  'rarity': 'Uncommon',
  'appearance': 'A delicate glass vial filled with a rich, golden liquid that shimmers softly in the light and has a faint scent of honey and rose petals.',
  'effect': 'Restores a moderate amount of health (typically 2d8 + 2 HP or equivalent) and grants a slight boost to vitality, allowing the drinker to ignore the first point of damage they take in the next hour.',
  'usage_instructions': 'Drink the entire contents of the vial slowly, feeling the soothing warmth spread through the body.',
  'duration': 'Instant healing, with the vitality boost lasting for 1 hour.',
  'side_effects': 'A gentle tingling sensation in the fingers and toes, and a subtle feeling of rejuvenation.',
  'origin_lore': 'Brewed by skilled herbalists who infuse the mixture with prayers to the gods of healing and protection,

Как видите, у нас достаточно много информации по каждому предмету. Представлено это двумя способами:
* структурированная информация в поле `structured`
* текстовое описание, которое в шутливой и несколько искаженной форме пересказывает одну и ту же информацию в поле `vague_description`.
Теперь вашей задачей будет создание двухэтапной рекомендательной системы, которая по запросу пользователя будет:
* Сначала выполните поиск по базе данных, чтобы найти 5-10 кандидатов,
* Затем запросите LLM, чтобы проанализировать результаты поиска и выбрать 2-3 окончательных кандидата. Таким образом, LLM будет играть роль **переранжировщика**. Подробнее о реранкерах мы поговорим в следующем блокноте.
Мы также предлагаем вам провести следующие эксперименты:
* Попробуйте и сравните поиск, использующий любую из трех баз данных:
  * База данных, состоящая только из расплывчатых описаний.
  * База данных, содержащая структурированные описания в формате JSON.
  * База данных, содержащая структурированные описания в чистом текстовом формате со всей удаленной разметкой JSON.
* Попробуйте сравнить извлечение с предварительной обработкой запроса и без нее. В таких сценариях часто бывает полезно переформулировать запрос, сделав его менее разговорным и более конкретным.
Мы обсудим автоматическую оценку RAG в одну из следующих недель. Прямо сейчас мы предлагаем просто собрать 10 различных запросов и внимательно изучить результаты.

In [ ]:
# <YOUR CODE HERE>

## Задача 2. Пробуем другую векторную базу данных — упражнение по разработке с помощью LLM
Существует множество векторных магазинов, и вам, возможно, придется использовать разные в зависимости от вкусов ваших коллег и других соображений. В этом задании вы попробуете [Qdrant](https://qdrant.tech/), которая также является удобной и эффективной базой данных.
Мы предлагаем вам использовать для этого упражнения клиент в памяти.
А поскольку никаких пояснений мы не даем, то лучшим способом справиться с этой задачей будет попросить LLM перевести приведенный выше код из LanceDB в Qdrant! Мы советуем использовать LLM, который также может выполнять веб-поиск (например, ChatGPT или Gemini), поскольку библиотеки могут меняться довольно быстро. Это не должно быть проблематично, но если у вас возникли проблемы с задачей, не стесняйтесь проверить наше решение. Однако будьте готовы к тому, что код может один или два раза дать сбой, прежде чем вы исправите его с помощью LLM.

Начнем с обычных приготовлений.

In [11]:
import os
import re

from tqdm import tqdm
from bs4 import BeautifulSoup
from markdown import markdown
from pathlib import Path


def markdown_to_text(markdown_string):
    """ Converts a markdown string to plaintext """

    # md -> html -> text since BeautifulSoup can extract text cleanly
    html = markdown(markdown_string)

    html = re.sub(r'<!--((.|\n)*)-->', '', html)
    html = re.sub('<code>bash', '<code>', html)

    # extract text
    soup = BeautifulSoup(html, "html.parser")
    text = ''.join(soup.findAll(text=True))

    text = re.sub('```(py|diff|python)', '', text)
    text = re.sub('```\n', '\n', text)
    text = re.sub('-         .*', '', text)
    text = text.replace('...', '')
    text = re.sub('\n(\n)+', '\n\n', text)

    return text


def prepare_files(input_dir="transformers/docs/source/en/", output_dir="docs"):
    # Convert string paths to Path objects
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    # Check if input directory exists
    assert input_dir.is_dir(), "Input directory doesn't exist"
    output_dir.mkdir(parents=True, exist_ok=True)

    for root, subdirs, files in tqdm(os.walk(input_dir)):
        root_path = Path(root)
        for file_name in files:
            file_path = root_path / file_name
            parent = root_path.stem if root_path.stem != input_dir.stem else ""

            if file_path.is_file():
                with open(file_path, encoding="utf-8") as f:
                    md = f.read()
                text = markdown_to_text(md)

                output_file = output_dir / f"{parent}_{Path(file_name).stem}.txt"
                with open(output_file, "w", encoding="utf-8") as f:
                    f.write(text)


И теперь ваша очередь загрузить данные в экземпляр базы данных Qdrant и соответствующим образом обновить функцию `answer_with_rag`!

In [ ]:
# <YOUR CODE HERE>

## Задача 3. Более безопасное устройство text2sql
Когда вы объединяете SQL и LLM, вы можете столкнуться с проблемой определения степени свободы, которую вы готовы предоставить агенту. Может ли он выполнять только запросы `SELECT`? Или вы разрешаете ему определенным образом изменять ваши таблицы? Чем больше свободы вы ему дадите, тем больше вы рискуете потерять данные или изменить их непредсказуемым образом.
Наш `text_to_sql_bot` использует
```
pd.read_sql_query(sql_query, self.db_conn)
```
для запуска запросов. Эта функция предназначена для выполнения запросов, которые возвращают строки (в основном это запросы `SELECT`), и она не сможет выполнить запрос, который манипулирует таблицами или данными (поэтому нет `DROP`, `DELETE`, `CREATE` и т. д.
Но что, если нашему SQL-боту действительно нужно изменить таблицу? Например, редактировать побочные эффекты на основе отзывов клиентов?
В этом задании вы попробуете — и по пути можете получить немного синяков. Что мы предлагаем вам сделать:
**Шаг 1.** Попробуйте обновить бота, внося как можно меньше разовых изменений. А именно:
- Измените `pd.read_sql_query(sql_query, self.db_conn)` на `query_db(self.db_conn, sql_query)` и немного ослабьте правила системных промптов, чтобы учесть некоторые изменения, а именно добавление побочных эффектов к зельям.
**Шаг 2.** Независимо от того, что вы пишете в системном промпте, вы фактически внесли хаос в свою жизнь, разрешив выполнение случайных запросов. Проверьте это, заставив бота удалять данные о покупках Торна. Thorne — наш клиент с наибольшими расходами; можешь использовать его ;)
Вспомните методы взлома из темы 1. Проявите всю свою хитрость и изобретательность! В конце концов, это будет не так сложно.
**Шаг 3.** Теперь попытайтесь снова обезопасить бота. Основной принцип здесь таков: вы не можете запретить LLM делать что-либо, кроме промптов. Итак, если вам нужна по-настоящему безопасная система, вам нужно выполнить жесткое программирование. Вот несколько возможных решений, которые вы могли бы изучить:
* Представляем простые проверки того, что `sql_query`, сгенерированный LLM, не содержит нецензурных слов, таких как `DELETE`, `DROP` и т. д.
* Добавление к этому уровня защиты LLM: для запросов, отличных от SELECT, он будет решать, являются ли они безопасными или вредоносными. Опять же, ни один LLM не является на 100% защищенным от сбоев, но обмануть LLM, ввод и вывод которого скрыты от вас, может быть намного сложнее.
* Вместо того, чтобы выполнять проверки после генерации SQL-запроса, вы можете сделать это в самом начале. Просто добавьте классификатор LLM, чтобы различать две ситуации:
  - Потенциальный оператор `SELECT` для которого будет вызываться безопасный `pd.read_sql_query(sql_query, self.db_conn)`
  — Запрос на добавление определенного побочного эффекта к конкретному зелью — его можно получить с помощью выходных данных JSON. Затем вы просто подключите их к готовому шаблону SQL-запроса!
Все это кажется не слишком захватывающим – мы как будто не верим в магию магистратуры… Но дело в том, что магия – это круто.но только до тех пор, пока ваши бизнес-процессы не пострадают от побочных эффектов.
Так что будьте осторожны и относитесь к возможностям магистратуры с долей скептицизма!

In [ ]:
# <YOUR CODE HERE>